# GPU Jacobian-lens prompt playground

This is the CUDA/GPU version of `02_fit_smoke_lens_cpu.ipynb`. It loads the local Qwen 2.5 3B model on one NVIDIA GPU and fits the smoke-test lens only when no saved GPU lens is available.

**Normal workflow**

1. Run sections 1–5 once.
2. Change only `evaluation_prompt` in the final cell.
3. Rerun only that final cell for every new prompt.

The model and fitted Jacobian stay in GPU memory while the kernel is alive. After a kernel restart you must reload the model, but the lens is loaded from `artifacts/lenses/` instead of being fitted again.

## Install CUDA-enabled PyTorch first

The current project environment may contain a CPU-only PyTorch build. **Stop the notebook kernel**, run one of these commands from the repository root, then restart Jupyter and select the project's `.venv` kernel. Do not run the installation after `torch` has already been imported into a live kernel.

Recommended stable CUDA 13.0 build for a recent NVIDIA driver:

```bash
uv pip install --python .venv/bin/python --reinstall "torch==2.12.0" --index-url https://download.pytorch.org/whl/cu130
```

Compatibility build for an older driver/GPU (CUDA 12.6):

```bash
uv pip install --python .venv/bin/python --reinstall "torch==2.12.0" --index-url https://download.pytorch.org/whl/cu126
```

Only `torch` is required by this project; `torchvision` and `torchaudio` are unnecessary. The wheel supplies CUDA runtime libraries, but the machine must still have an NVIDIA GPU and a compatible NVIDIA driver. See the [official PyTorch installation selector](https://docs.pytorch.org/get-started/locally/) and [PyTorch 2.12 wheel matrix](https://pytorch.org/get-started/previous-versions/#v2120) if you need a different platform build.

## 1. Imports and configuration

Keep `FORCE_REFIT = False` during normal prompt testing. `FIT_DIM_BATCH = None` chooses a conservative value from GPU VRAM; set an integer manually if desired. If fitting runs out of memory, try `4`, `2`, or `1`.

In [ ]:
import logging
import sys
import time
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the repository root whether Jupyter starts here or in notebooks/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "jlens").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the jacobian-lens repository root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import jlens
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = PROJECT_ROOT / "models/Qwen2.5-3B-Instruct"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/lenses"
CHECKPOINT_PATH = ARTIFACT_DIR / "qwen3b_gpu_smoke_fit_checkpoint.pt"
LENS_PATH = ARTIFACT_DIR / "qwen3b_gpu_smoke_lens.pt"

GPU_INDEX = 0
FORCE_REFIT = False
FIT_DIM_BATCH = None  # None = choose automatically from total GPU VRAM.

FIT_PROMPTS = [
    (
        "Artificial intelligence systems process natural language by "
        "transforming input tokens through multiple neural network layers. "
        "Intermediate representations encode different types of information "
        "before the model predicts the next token."
    )
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Model path   : {MODEL_PATH}")
print(f"Lens path    : {LENS_PATH}")
print(f"Saved lens   : {LENS_PATH.exists()}")

## 2. Verify and configure CUDA

This notebook intentionally stops if CUDA is unavailable. It never silently runs the 3B model on CPU.

In [ ]:
print(f"Torch version      : {torch.__version__}")
print(f"Torch CUDA build   : {torch.version.cuda}")
print(f"CUDA available     : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Stop this kernel, install a CUDA-enabled PyTorch "
        "build using the command above, confirm the NVIDIA driver is installed, "
        "then restart Jupyter."
    )
if GPU_INDEX >= torch.cuda.device_count():
    raise ValueError(
        f"GPU_INDEX={GPU_INDEX}, but only {torch.cuda.device_count()} CUDA GPU(s) exist."
    )

DEVICE = torch.device(f"cuda:{GPU_INDEX}")
torch.cuda.set_device(DEVICE)
gpu = torch.cuda.get_device_properties(DEVICE)
total_vram_gib = gpu.total_memory / 1024**3
MODEL_DTYPE = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

if FIT_DIM_BATCH is None:
    if total_vram_gib >= 40:
        fit_dim_batch = 32
    elif total_vram_gib >= 24:
        fit_dim_batch = 16
    elif total_vram_gib >= 16:
        fit_dim_batch = 8
    elif total_vram_gib >= 12:
        fit_dim_batch = 4
    else:
        fit_dim_batch = 1
else:
    fit_dim_batch = int(FIT_DIM_BATCH)
    if fit_dim_batch < 1:
        raise ValueError("FIT_DIM_BATCH must be at least 1.")

print(f"GPU count          : {torch.cuda.device_count()}")
print(f"Selected GPU       : {GPU_INDEX} — {gpu.name}")
print(f"Total VRAM         : {total_vram_gib:.2f} GiB")
print(f"Model dtype        : {MODEL_DTYPE}")
print(f"Fit dim batch      : {fit_dim_batch}")

## 3. Load and wrap the model once

Qwen is loaded in BF16 on supported GPUs, otherwise FP16. Rerunning this cell with the same path, device, and dtype reuses the existing objects instead of allocating a second model.

In [ ]:
model_key = (str(MODEL_PATH.resolve()), str(DEVICE), str(MODEL_DTYPE))
model_is_ready = all(
    name in globals() for name in ("tokenizer", "hf_model", "model")
)

if model_is_ready and globals().get("_loaded_model_key") == model_key:
    print("Model is already loaded on this GPU; reusing it.")
else:
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
    )

    print(f"Loading Qwen 3B as {MODEL_DTYPE} on {DEVICE}...")
    load_started = time.perf_counter()
    hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        dtype=MODEL_DTYPE,
        local_files_only=True,
        low_cpu_mem_usage=True,
    )
    hf_model.to(DEVICE)
    hf_model.eval()
    torch.cuda.synchronize(DEVICE)

    print(f"Model loaded in {time.perf_counter() - load_started:.2f} seconds")
    print("Wrapping the model with jlens.from_hf()...")
    model = jlens.from_hf(hf_model, tokenizer, compile=False)
    _loaded_model_key = model_key

source_layer = model.n_layers - 2
target_layer = model.n_layers - 1
allocated_gib = torch.cuda.memory_allocated(DEVICE) / 1024**3
reserved_gib = torch.cuda.memory_reserved(DEVICE) / 1024**3

print(model)
print(f"Number of layers : {model.n_layers}")
print(f"Residual width   : {model.d_model}")
print(f"Source layer     : {source_layer}")
print(f"Target layer     : {target_layer}")
print(f"GPU allocated    : {allocated_gib:.2f} GiB")
print(f"GPU reserved     : {reserved_gib:.2f} GiB")

## 4. Load the saved GPU lens, or fit it once

The GPU notebook uses separate `qwen3b_gpu_smoke_*` artifacts. Once fitted, later sessions load the saved lens. The final Jacobian matrix is also cached on the selected GPU for repeated evaluations.

If this cell raises CUDA out-of-memory during fitting, restart the kernel, set `FIT_DIM_BATCH` to a smaller value in section 1, and run again. The checkpoint allows an interrupted normal fit to resume.

In [ ]:
if LENS_PATH.is_file() and not FORCE_REFIT:
    print(f"Loading existing lens: {LENS_PATH}")
    lens = jlens.JacobianLens.load(str(LENS_PATH))
else:
    if FORCE_REFIT:
        print("FORCE_REFIT=True: fitting the lens from scratch.")
    else:
        print("No saved GPU lens found; fitting it now.")

    print("Smoke fitting configuration:")
    print(f"  Prompts      : {len(FIT_PROMPTS)}")
    print("  Max seq len  : 24")
    print("  Skip first   : 2")
    print(f"  Dim batch    : {fit_dim_batch}")
    print(
        f"  Backward passes per prompt: "
        f"{(model.d_model + fit_dim_batch - 1) // fit_dim_batch}"
    )

    fit_started = time.perf_counter()
    lens = jlens.fit(
        model,
        prompts=FIT_PROMPTS,
        source_layers=[source_layer],
        target_layer=target_layer,
        dim_batch=fit_dim_batch,
        max_seq_len=24,
        skip_first=2,
        checkpoint_path=str(CHECKPOINT_PATH),
        checkpoint_every=1,
        resume=not FORCE_REFIT,
    )
    torch.cuda.synchronize(DEVICE)
    print(f"Fitting completed in {time.perf_counter() - fit_started:.2f} seconds")
    lens.save(str(LENS_PATH))
    print(f"Lens saved to: {LENS_PATH}")

if lens.d_model != model.d_model:
    raise ValueError(
        f"Lens width {lens.d_model} does not match model width {model.d_model}."
    )
if source_layer not in lens.source_layers:
    raise ValueError(
        f"Layer {source_layer} is not in saved lens layers {lens.source_layers}."
    )

# JacobianLens.load() maps tensors to CPU. Cache them on the GPU once so
# repeated prompt evaluations do not copy the matrix on every call.
lens.jacobians = {
    layer: jacobian.to(DEVICE) for layer, jacobian in lens.jacobians.items()
}

print(f"Ready: {lens}")
print(f"Lens size: {LENS_PATH.stat().st_size / 1024**2:.2f} MiB")
print(f"Lens device: {lens.jacobians[source_layer].device}")

## 5. Define the reusable evaluation helper

Run this once. By default each evaluation compares the Jacobian lens, vanilla logit lens, and final model logits. Set `compare_logit_lens=False` in the final cell when you only need J-lens and model results; that avoids the second forward pass.

In [ ]:
def decode_top_tokens(
    logits: torch.Tensor,
    top_k: int = 10,
) -> list[tuple[str, float]]:
    """Decode the highest-scoring tokens from one logits vector."""
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    values, indices = logits.topk(min(top_k, logits.shape[-1]))
    return [
        (repr(tokenizer.decode([token_id])), score)
        for token_id, score in zip(
            indices.tolist(),
            values.tolist(),
            strict=True,
        )
    ]


def print_top_tokens(title: str, logits: torch.Tensor, top_k: int) -> None:
    print(f"\n{title}")
    for token, score in decode_top_tokens(logits, top_k=top_k):
        print(f"{token:>20}  {score:>10.4f}")


def evaluate_prompt(
    prompt: str,
    *,
    top_k: int = 10,
    position: int = -1,
    max_seq_len: int = 512,
    compare_logit_lens: bool = True,
) -> dict[str, object]:
    """Evaluate one prompt without reloading the model or refitting the lens."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string.")

    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    started = time.perf_counter()

    jlens_logits, model_logits, input_ids = lens.apply(
        model,
        prompt,
        layers=[source_layer],
        positions=[position],
        max_seq_len=max_seq_len,
        use_jacobian=True,
    )

    logit_lens_logits = None
    if compare_logit_lens:
        logit_lens_logits, _, _ = lens.apply(
            model,
            prompt,
            layers=[source_layer],
            positions=[position],
            max_seq_len=max_seq_len,
            use_jacobian=False,
        )

    torch.cuda.synchronize(DEVICE)
    elapsed_seconds = time.perf_counter() - started
    peak_allocated_gib = torch.cuda.max_memory_allocated(DEVICE) / 1024**3

    print("=== Evaluation prompt ===")
    print(prompt)
    print("\nInput tokens:")
    print(tokenizer.convert_ids_to_tokens(input_ids[0].tolist()))

    print_top_tokens(
        f"J-lens top tokens at layer {source_layer}:",
        jlens_logits[source_layer][0],
        top_k,
    )
    if logit_lens_logits is not None:
        print_top_tokens(
            f"Logit-lens top tokens at layer {source_layer}:",
            logit_lens_logits[source_layer][0],
            top_k,
        )
    print_top_tokens("Model final top tokens:", model_logits[0], top_k)
    print(f"\nEvaluation completed in {elapsed_seconds:.2f} seconds")
    print(f"Peak GPU memory allocated: {peak_allocated_gib:.2f} GiB")

    return {
        "prompt": prompt,
        "input_ids": input_ids,
        "jlens_logits": jlens_logits,
        "logit_lens_logits": logit_lens_logits,
        "model_logits": model_logits,
        "elapsed_seconds": elapsed_seconds,
        "peak_allocated_gib": peak_allocated_gib,
    }


print("Evaluation helper is ready. Rerun only the final cell for new prompts.")

## 6. Prompt evaluation — rerun only this cell

Edit `evaluation_prompt` and press **Shift+Enter**. Nothing is reloaded or refitted.

In [ ]:
evaluation_prompt = "The largest planet in the solar system is"

evaluation_result = evaluate_prompt(
    evaluation_prompt,
    top_k=10,
    position=-1,
    compare_logit_lens=True,
)